# GLUE RTE with src-backed Custom BERT

This notebook mirrors the original `BERT_glue_rte.ipynb` flow, but it imports the custom transformer implementation from `src/ptq_tr/models/nlp/custom_bert.py` instead of defining the BERT stack inline.


In [ ]:
# Optional once per environment
# %pip install "huggingface-hub>=0.34.0,<1.0" "transformers>=4.40,<4.46" datasets evaluate tqdm


In [ ]:
import sys
from pathlib import Path

import torch
from datasets import load_dataset
import evaluate
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

repo_root = Path.cwd()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from ptq_tr.models.nlp.custom_bert import CustomBertModel, CustomBertForRTE
from ptq_tr.quantization.modules import (
    IntGeluTS,
    IntSoftmaxTS,
    QLayerNorm,
    QuantizedLinear,
    QuantizedMatmul,
    qHadamardProd,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
model_name = "textattack/bert-base-uncased-RTE"

tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)

# Hugging Face reference checkpoint for weight loading / comparison
hf_model = AutoModelForSequenceClassification.from_pretrained(model_name)
hf_model = hf_model.to(device).eval()

# Custom BERT implementation imported from src
model = CustomBertForRTE(config)
load_result = model.load_state_dict(hf_model.state_dict(), strict=False)
model = model.to(device).eval()

print("Custom encoder class:", CustomBertModel)
print("Missing keys:", len(load_result.missing_keys))
print("Unexpected keys:", len(load_result.unexpected_keys))
print("Missing examples:", load_result.missing_keys[:20])
print("id2label:", model.config.id2label)


In [ ]:
# Swap quantized modules in exactly as in the original notebook,
# but the modules now live in src instead of notebook cells.
# q_module_list = [QLayerNorm, IntSoftmaxTS, QuantizedLinear, QuantizedMatmul, IntGeluTS, qHadamardProd]
# q_module_list = [QLayerNorm, IntSoftmaxTS, QuantizedLinear, QuantizedMatmul]
q_module_list = [QLayerNorm]

model.set_q_module_list(q_module_list)
model.set_quant()

def audit_modes(model):
    note = []
    for name, module in model.named_modules():
        quant_flag = getattr(module, "quant", None)
        opt_flag = getattr(module, "is_opt_scale", None)
        if quant_flag is True or opt_flag is True:
            note.append((name, type(module).__name__, quant_flag, opt_flag))
    return note

note = audit_modes(model)
print("modules not in pure-float mode:", len(note))
print(*note[:50], sep="\n")


In [ ]:
train_stream = load_dataset(
    "glue",
    "rte",
    split="train",
    streaming=True,
)

model.eval()

if (
    QuantizedLinear in q_module_list
    or QuantizedMatmul in q_module_list
    or IntSoftmaxTS in q_module_list
    or QLayerNorm in q_module_list
):
    with torch.no_grad():
        model.set_calibration_flag()
        for i, sample in enumerate(train_stream):
            if i == 100:
                break

            s1 = sample["sentence1"]
            s2 = sample["sentence2"]
            if s1 is None or s2 is None:
                continue

            inputs = tokenizer(
                s1,
                s2,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128,
            )

            token_type_ids = inputs.get("token_type_ids")
            _ = model(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device),
                token_type_ids=token_type_ids.to(device) if token_type_ids is not None else None,
            )

        model.unset_calibration_flag()
        print("Quantization parameters were set")


In [ ]:
premise_entail = "William Shakespeare wrote the play Hamlet."
hypothesis_entail = "Shakespeare is the author of Hamlet."

premise_not_entail = "A soccer match was played in an empty stadium."
hypothesis_not_entail = "Thousands of fans attended the soccer match."

def run_rte_example(premise, hypothesis):
    inputs = tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    )
    token_type_ids = inputs.get("token_type_ids")

    with torch.no_grad():
        outputs = model(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device),
            token_type_ids=token_type_ids.to(device) if token_type_ids is not None else None,
        )

    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1)
    pred = int(torch.argmax(logits, dim=-1).item())
    return {
        "premise": premise,
        "hypothesis": hypothesis,
        "prediction_id": pred,
        "label": model.config.id2label[pred],
        "probabilities": probs.cpu().numpy(),
    }

res_entail = run_rte_example(premise_entail, hypothesis_entail)
res_not_entail = run_rte_example(premise_not_entail, hypothesis_not_entail)

print(res_entail)
print()
print(res_not_entail)


In [ ]:
BATCH_SIZE = 16
MAX_LEN = 128
NUM_VAL_EXAMPLES = 277

val_stream = load_dataset(
    "glue",
    "rte",
    split="validation",
    streaming=True,
).shuffle(seed=400)
streamer = val_stream.iter(batch_size=BATCH_SIZE)
metric = evaluate.load("glue", "rte")

predictions = []
references = []

def get_batch_predictions(batch):
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt",
    )
    token_type_ids = inputs.get("token_type_ids")

    with torch.no_grad():
        outputs = model(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device),
            token_type_ids=token_type_ids.to(device) if token_type_ids is not None else None,
        )

    return torch.argmax(outputs.logits, dim=-1).cpu().tolist()

for i, batch in enumerate(tqdm(streamer, total=NUM_VAL_EXAMPLES // BATCH_SIZE)):
    if i * BATCH_SIZE >= NUM_VAL_EXAMPLES:
        break

    valid_idx = [
        j
        for j, (a, b) in enumerate(zip(batch["sentence1"], batch["sentence2"]))
        if a is not None and b is not None
    ]
    if not valid_idx:
        continue

    filtered_batch = {
        "sentence1": [batch["sentence1"][j] for j in valid_idx],
        "sentence2": [batch["sentence2"][j] for j in valid_idx],
        "label": [batch["label"][j] for j in valid_idx],
    }

    batch_preds = get_batch_predictions(filtered_batch)
    predictions.extend(batch_preds)
    references.extend(filtered_batch["label"])

    if (i + 1) % 10 == 0:
        res = metric.compute(predictions=predictions, references=references)
        print(f"[Batch {i + 1}] Accuracy: {res['accuracy']:.4f}")

res = metric.compute(predictions=predictions, references=references)
print(f"Final Results | Accuracy: {res['accuracy']:.4f}")
